# Week 2-3 · conditional edge로 티켓 경로 나누기

## 시나리오
일반 billing 문의는 billing queue로, urgent 기술 장애는 incident escalation으로 실제 graph 경로를 나눕니다.

## 학습 목표
- 분류 결과를 state에 기록하는 node를 만든다.
- `choose_route`가 state에서 route key를 계산한다.
- `add_conditional_edges`로 서로 다른 node를 실행한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · state와 node

In [ ]:
# 실행 순서: 1단계 · state와 node에서 PracticeTicketState, classify_node, billing_queue을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · state와 node.
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

# 분류와 routing node가 공유할 입력·판단·trace 필드를 선언합니다.
class PracticeTicketState(TypedDict):
    subject: str
    category: str
    priority: str
    route: str
    steps: list[str]

# subject를 category와 priority로 바꾸고 실행 trace에 classify를 기록합니다.
def classify_node(state: PracticeTicketState) -> dict:
    text = state["subject"].lower()
    if "outage" in text:
        return {"category": "technical", "priority": "urgent", "steps": state["steps"] + ["classify"]}
    return {"category": "billing", "priority": "normal", "steps": state["steps"] + ["classify"]}

# 일반 billing 문의가 도착한 queue를 state에 기록할 뿐 외부 조치는 하지 않습니다.
def billing_queue(state: PracticeTicketState) -> dict:
    return {"route": "billing_queue", "steps": state["steps"] + ["billing_queue"]}

# 긴급 기술 문의를 incident 경로로 표시하되 실제 escalation은 실행하지 않습니다.
def incident_escalation(state: PracticeTicketState) -> dict:
    return {"route": "incident_escalation", "steps": state["steps"] + ["incident_escalation"]}

# priority를 graph의 route key로 변환해 다음 edge를 결정합니다.
def choose_route(state: PracticeTicketState) -> str:
    return "incident" if state["priority"] == "urgent" else "billing"

### 2단계 · conditional graph 구성

In [ ]:
# 실행 순서: 2단계 · conditional graph 구성에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · conditional graph 구성.
practice_builder = StateGraph(PracticeTicketState)
practice_builder.add_node("classify", classify_node)
practice_builder.add_node("billing_queue", billing_queue)
practice_builder.add_node("incident_escalation", incident_escalation)
practice_builder.add_edge(START, "classify")
practice_builder.add_conditional_edges("classify", choose_route, {
    "billing": "billing_queue",
    "incident": "incident_escalation",
})
practice_builder.add_edge("billing_queue", END)
practice_builder.add_edge("incident_escalation", END)
practice_graph = practice_builder.compile()

### 3단계 · 두 경로 invoke

In [ ]:
# 실행 순서: 3단계 · 두 경로 invoke에서 practice_input을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 두 경로 invoke.
# 각 graph 실행이 같은 빈 state에서 시작하도록 입력 fixture를 만듭니다.
def practice_input(subject: str) -> PracticeTicketState:
    return {"subject": subject, "category": "", "priority": "", "route": "", "steps": []}

billing_result = practice_graph.invoke(practice_input("Duplicate invoice"))
outage_result = practice_graph.invoke(practice_input("Global outage"))
assert billing_result["steps"] == ["classify", "billing_queue"]
assert outage_result["steps"] == ["classify", "incident_escalation"]
{"billing": billing_result, "outage": outage_result}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
router가 모르는 값을 반환하면 graph는 진행하지 않아야 합니다. 어떤 route node도 환불·계정 변경·ticket close를 실행하지 않습니다.

## 실제 app 연결
Week 2 app은 이 패턴 앞에 structured classification과 filtered retrieval을 연결합니다. Week 1의 선형 graph와 달리 이번에는 state에 따라 다음 edge가 달라집니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `04_ticket_workflow_evaluation.ipynb`에서는 분류·검색·근거 gate·route를 하나의 최소 pipeline으로 평가합니다.